# Diagnóstico de la Brecha Digital e Inclusión Tecnológica en Argentina

**Práctica Profesional I**  
**Alumno:** Natan Levi Rabinovich  
**Carrera:** Técnico Superior en Ciencia de Datos e Inteligencia Artificial  

Este notebook contiene el proceso completo de exploración, limpieza, análisis estadístico y visualización del dataset utilizado en el proyecto.

> **Importante:** el dataset fue generado de forma sintética con fines académicos. Los resultados no representan estimaciones reales de la población argentina.


## 1. Librerías y preparación del entorno

Primero importo las librerías que necesito para trabajar con los datos y generar los gráficos. También creo la carpeta `graficos` para guardar las imágenes generadas durante el análisis.


In [ ]:
# Importo las librerias necesarias para el analisis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Creo la carpeta donde voy a guardar los graficos.
# Si ya existe, no realiza ningun cambio.
os.makedirs("graficos", exist_ok=True)


## 2. Carga del dataset en Google Colab

Al ejecutar la siguiente celda, Google Colab permite seleccionar el archivo CSV desde la computadora. Para este proyecto el archivo esperado es `eph_tecnologia_microdatos.csv`.


In [ ]:
from google.colab import files

# Selecciono el archivo CSV desde mi computadora
uploaded = files.upload()

# Tomo el nombre del primer archivo cargado para no depender de un nombre fijo
nombre_archivo = next(iter(uploaded))

# Cargo el dataset original con pandas
df = pd.read_csv(nombre_archivo)

print(f"Archivo cargado: {nombre_archivo}")


## 3. Exploración inicial de los datos

Antes de modificar los datos reviso la estructura general del dataset, sus dimensiones, tipos de variables, valores faltantes y categorías disponibles.


In [ ]:
# Muestro las dimensiones para saber cuantas filas y columnas tiene el dataset
print("\nDIMENSIONES DEL DATASET:")
print(df.shape)

# Visualizo las primeras 3 filas para conocer la estructura
# y ver ejemplos de los datos de cada columna
print("\nPRIMERAS FILAS:")
print(df.head(3))

# Muestro informacion general: columnas, datos no nulos y tipo de dato
print("\nINFORMACION DEL DATASET:")
df.info()

# Cuento cuantos valores nulos (NaN) existen en cada columna
print("\nVALORES NULOS:")
print(df.isnull().sum())

# Muestro los valores posibles de las principales variables categoricas
print("\nVALORES DE REGION:")
print(df["REGION"].value_counts().sort_index())

print("\nVALORES DE NIVEL EDUCATIVO:")
print(df["NIVEL_ED"].value_counts().sort_index())

print("\nVALORES DE TENENCIA DE COMPUTADORA:")
print(df["IV12_3"].value_counts(dropna=False).sort_index())

print("\nVALORES DE INTERNET:")
print(df["II7"].value_counts(dropna=False).sort_index())


### Resultado de la exploración

En el dataset existen valores `NaN` y también respuestas codificadas con `9`, que significan **Sin respuesta**. Estos códigos deben tratarse como datos faltantes para evitar que sean interpretados como valores reales durante los cálculos.


## 4. Limpieza y recodificación

Los códigos `9` de `NIVEL_ED`, `IV12_3` e `II7` se convierten a `NaN`. No elimino automáticamente filas completas porque un hogar puede tener información faltante en una variable y seguir siendo útil para otros análisis.


In [ ]:
# Creo una copia para conservar el dataset original
# sin modificaciones
df_clean = df.copy()

# Reemplazo los codigos 9 (Sin respuesta) por NaN
# para evitar que sean tomados como valores reales
df_clean["NIVEL_ED"] = df_clean["NIVEL_ED"].replace(9, np.nan)
df_clean["IV12_3"] = df_clean["IV12_3"].replace(9, np.nan)
df_clean["II7"] = df_clean["II7"].replace(9, np.nan)

# Visualizo los valores faltantes luego de la recodificacion
print("\nVALORES NULOS DESPUES DE LA RECODIFICACION:")
print(df_clean.isnull().sum())

# Compruebo que los codigos 9 hayan sido recodificados correctamente
print("\nVALORES DE NIVEL EDUCATIVO DESPUES DE LIMPIAR:")
print(df_clean["NIVEL_ED"].value_counts(dropna=False).sort_index())

print("\nVALORES DE COMPUTADORA DESPUES DE LIMPIAR:")
print(df_clean["IV12_3"].value_counts(dropna=False).sort_index())

print("\nVALORES DE INTERNET DESPUES DE LIMPIAR:")
print(df_clean["II7"].value_counts(dropna=False).sort_index())


### Decisión sobre los valores faltantes

Para `ITF` se podría haber utilizado una medida de tendencia central para completar los datos faltantes. Como los ingresos presentan asimetría positiva y valores extremos, la **mediana** sería una mejor opción que la media porque es menos sensible a estos valores.

Sin embargo, en este proyecto se decidió conservar los `NaN` y trabajar en cada análisis solamente con los registros válidos. Para variables categóricas podría utilizarse la **moda**, pero esto aumentaría artificialmente la categoría que más se repite.


In [ ]:
# Exporto el dataset luego del proceso de limpieza y recodificacion,
# conservando el archivo original sin modificaciones
df_clean.to_csv("df_clean.csv", index=False)

print("Dataset limpio generado correctamente.")
print("Dimensiones:", df_clean.shape)


## 5. Estadística descriptiva

Calculo estadísticas descriptivas de las principales variables numéricas:

- **Media y mediana:** medidas de tendencia central.
- **Mínimo y máximo:** permiten conocer los extremos.
- **Desvío estándar:** permite analizar la dispersión.

La moda no se utiliza en estas variables numéricas porque no aporta información relevante para el objetivo del análisis.


In [ ]:
print("\n--- ESTADISTICAS DESCRIPTIVAS ---")

variables_numericas = ["IX_TOT", "ITF", "DECIFR"]

for variable in variables_numericas:
    print(f"\nVariable: {variable}")
    print(f"Media: {df_clean[variable].mean():.2f}")
    print(f"Mediana: {df_clean[variable].median():.2f}")
    print(f"Minimo: {df_clean[variable].min():.2f}")
    print(f"Maximo: {df_clean[variable].max():.2f}")
    print(f"Desvio estandar: {df_clean[variable].std():.2f}")

# Calculo los percentiles 25, 50 y 75 del ingreso familiar
# para conocer mejor como se distribuyen los ingresos
print("\n--- PERCENTILES DE INGRESO FAMILIAR ---")
print(df_clean["ITF"].quantile([0.25, 0.50, 0.75]))


## 6. Distribución del Ingreso Total Familiar

Primero utilizo un histograma para observar la distribución de `ITF`. Luego utilizo un boxplot para analizar la dispersión e identificar posibles valores extremos.


In [ ]:
# Histograma del Ingreso Total Familiar
plt.figure(figsize=(10, 6))
plt.hist(
    df_clean["ITF"].dropna(),
    bins=30,
    edgecolor="black"
)
plt.title("Distribucion del Ingreso Familiar")
plt.xlabel("Ingreso Total Familiar")
plt.ylabel("Cantidad de hogares")
plt.tight_layout()
plt.savefig("graficos/histograma_ingresos.png", dpi=300, bbox_inches="tight")
plt.show()

# Boxplot para observar dispersion y posibles valores extremos
plt.figure(figsize=(10, 4))
plt.boxplot(
    df_clean["ITF"].dropna(),
    vert=False
)
plt.title("Distribucion del Ingreso Total Familiar")
plt.xlabel("Ingreso Total Familiar")
plt.tight_layout()
plt.savefig("graficos/boxplot_ingresos.png", dpi=300, bbox_inches="tight")
plt.show()


Los gráficos muestran una distribución con **asimetría positiva**: existe una mayor concentración de hogares en ingresos bajos y medios, con una cola que se extiende hacia los ingresos más altos. No se afirma curtosis sin calcularla específicamente.


# 7. Análisis bivariado

A continuación se analizan las tres hipótesis principales mediante tablas de contingencia, porcentajes y gráficos.


## 7.1 Hipótesis 1: ingreso y computadora

**Hipótesis:** los hogares de mayores ingresos presentan una mayor tenencia de computadora o laptop.


In [ ]:
# Selecciono solamente los registros que tienen valores validos
# tanto en DECIFR como en IV12_3
df_h1 = df_clean.dropna(subset=["DECIFR", "IV12_3"]).copy()

# Creo una tabla de contingencia
tabla_pc = pd.crosstab(
    df_h1["DECIFR"],
    df_h1["IV12_3"]
)

print("--- DECIL DE INGRESO VS TENENCIA DE COMPUTADORA ---")
display(tabla_pc)

# Calculo porcentajes dentro de cada decil
porcentaje_pc = pd.crosstab(
    df_h1["DECIFR"],
    df_h1["IV12_3"],
    normalize="index"
) * 100

print("--- PORCENTAJES POR DECIL ---")
display(porcentaje_pc.round(2))


In [ ]:
# Grafico el porcentaje de hogares con computadora por decil
pc_por_decil = porcentaje_pc[1.0]

plt.figure(figsize=(10, 6))
plt.plot(
    pc_por_decil.index,
    pc_por_decil.values,
    marker="o"
)
plt.title("Tenencia de computadora segun decil de ingreso")
plt.xlabel("Decil de ingreso")
plt.ylabel("Hogares con computadora (%)")
plt.xticks(range(1, 11))
plt.ylim(0, 100)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("graficos/computadora_por_decil.png", dpi=300, bbox_inches="tight")
plt.show()


**Resultado:** en el dataset analizado la tenencia de computadora pasa de aproximadamente **40,66% en el decil 1** a **91,30% en el decil 10**. La tendencia general respalda la Hipótesis 1.


## 7.2 Hipótesis 2: nivel educativo e Internet

**Hipótesis:** los hogares con mayor nivel educativo presentan un mayor acceso a Internet.


In [ ]:
# Selecciono los registros que tienen valores validos
# en NIVEL_ED e II7
df_h2 = df_clean.dropna(subset=["NIVEL_ED", "II7"]).copy()

# Tabla de contingencia
tabla_internet_educacion = pd.crosstab(
    df_h2["NIVEL_ED"],
    df_h2["II7"]
)

print("--- NIVEL EDUCATIVO VS ACCESO A INTERNET ---")
display(tabla_internet_educacion)

# Porcentajes por nivel educativo
porcentaje_internet_educacion = pd.crosstab(
    df_h2["NIVEL_ED"],
    df_h2["II7"],
    normalize="index"
) * 100

print("--- PORCENTAJES DE INTERNET POR NIVEL EDUCATIVO ---")
display(porcentaje_internet_educacion.round(2))


In [ ]:
# Grafico el porcentaje de hogares con Internet por nivel educativo
internet_por_educacion = porcentaje_internet_educacion[1.0]

niveles = [
    "Primaria\nincompleta",
    "Primaria\ncompleta",
    "Secundaria\nincompleta",
    "Secundaria\ncompleta",
    "Superior/\nUniversitaria"
]

plt.figure(figsize=(10, 6))
plt.bar(niveles, internet_por_educacion.values)
plt.title("Acceso a Internet segun nivel educativo")
plt.xlabel("Nivel educativo")
plt.ylabel("Hogares con Internet (%)")
plt.ylim(0, 100)
plt.tight_layout()
plt.savefig("graficos/internet_por_educacion.png", dpi=300, bbox_inches="tight")
plt.show()


**Resultado:** el acceso a Internet aumenta desde aproximadamente **43,90% en primaria incompleta** hasta **97,71% en superior/universitaria completa**. Los resultados respaldan la Hipótesis 2.


## 7.3 Hipótesis 3: ingreso e Internet

**Hipótesis:** los hogares de mayores ingresos presentan un mayor acceso a Internet.


In [ ]:
# Selecciono los registros con valores validos en DECIFR e II7
df_h3 = df_clean.dropna(subset=["DECIFR", "II7"]).copy()

# Creo una tabla de contingencia
tabla_internet_ingreso = pd.crosstab(
    df_h3["DECIFR"],
    df_h3["II7"]
)

print("--- DECIL DE INGRESO VS ACCESO A INTERNET ---")
display(tabla_internet_ingreso)

porcentaje_internet_ingreso = pd.crosstab(
    df_h3["DECIFR"],
    df_h3["II7"],
    normalize="index"
) * 100

print("--- PORCENTAJES DE INTERNET POR DECIL ---")
display(porcentaje_internet_ingreso.round(2))


In [ ]:
# Grafico el acceso a Internet segun decil de ingreso
internet_por_decil = porcentaje_internet_ingreso[1.0]

plt.figure(figsize=(10, 6))
plt.plot(
    internet_por_decil.index,
    internet_por_decil.values,
    marker="o"
)
plt.title("Acceso a Internet segun decil de ingreso")
plt.xlabel("Decil de ingreso")
plt.ylabel("Hogares con Internet (%)")
plt.xticks(range(1, 11))
plt.ylim(0, 100)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("graficos/internet_por_decil.png", dpi=300, bbox_inches="tight")
plt.show()


**Resultado:** el acceso a Internet pasa de aproximadamente **57,14% en el decil 1** a **95,70% en el decil 10**. Aunque existen pequeñas variaciones entre algunos deciles consecutivos, la tendencia general respalda la Hipótesis 3.


## 8. Análisis complementario por región

Creo un diccionario para relacionar cada código numérico con el nombre de su región. Luego comparo, por separado, el acceso a Internet y la tenencia de computadora.


In [ ]:
# Relaciono cada codigo de REGION con su nombre
regiones = {
    1: "GBA",
    40: "NOA",
    41: "NEA",
    42: "Cuyo",
    43: "Pampeana",
    44: "Patagonia"
}

df_region = df_clean.copy()
df_region["REGION_NOMBRE"] = df_region["REGION"].map(regiones)

# Tabla de contingencia: acceso a Internet por region
internet_region = pd.crosstab(
    df_region["REGION_NOMBRE"],
    df_region["II7"],
    normalize="index"
) * 100

print("--- ACCESO A INTERNET POR REGION ---")
display(internet_region.round(2))

# Tabla de contingencia: tenencia de computadora por region
pc_region = pd.crosstab(
    df_region["REGION_NOMBRE"],
    df_region["IV12_3"],
    normalize="index"
) * 100

print("--- TENENCIA DE COMPUTADORA POR REGION ---")
display(pc_region.round(2))


En este dataset las diferencias entre regiones son relativamente pequeñas en comparación con las diferencias observadas según ingreso y nivel educativo.


## 9. Matriz de correlación

Para facilitar la interpretación, las variables tecnológicas se recodifican como binarias:

- `1` = tiene acceso.
- `0` = no tiene acceso.

La matriz se utiliza como análisis complementario. Una correlación no significa necesariamente que una variable sea la causa de la otra.


In [ ]:
# Creo una copia para preparar las variables de la matriz
df_corr = df_clean.copy()

# Recodifico las variables tecnologicas:
# 1 = tiene acceso / 0 = no tiene acceso
df_corr["TIENE_PC"] = df_corr["IV12_3"].map({1: 1, 2: 0})
df_corr["TIENE_INTERNET"] = df_corr["II7"].map({1: 1, 2: 0})

variables_correlacion = [
    "IX_TOT",
    "NIVEL_ED",
    "ITF",
    "DECIFR",
    "TIENE_PC",
    "TIENE_INTERNET"
]

matriz_correlacion = df_corr[variables_correlacion].corr()

print("--- MATRIZ DE CORRELACION ---")
display(matriz_correlacion.round(2))


In [ ]:
# Grafico la matriz de correlacion utilizando una escala de colores
plt.figure(figsize=(9, 7))
plt.imshow(
    matriz_correlacion,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)
plt.colorbar(label="Correlacion")

plt.xticks(
    range(len(variables_correlacion)),
    variables_correlacion,
    rotation=45,
    ha="right"
)
plt.yticks(
    range(len(variables_correlacion)),
    variables_correlacion
)

# Muestro el valor de cada correlacion dentro del grafico
for i in range(len(variables_correlacion)):
    for j in range(len(variables_correlacion)):
        plt.text(
            j,
            i,
            f"{matriz_correlacion.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

plt.title("Matriz de correlacion")
plt.tight_layout()
plt.savefig("graficos/matriz_correlacion.png", dpi=300, bbox_inches="tight")
plt.show()


## 10. Conclusiones

A partir del análisis realizado:

1. Los hogares de mayores ingresos presentan una mayor tenencia de computadora.
2. El nivel educativo muestra una relación clara con el acceso a Internet.
3. Los hogares de mayores ingresos presentan un mayor acceso a Internet.
4. Las diferencias regionales fueron menores que las encontradas por ingreso y educación.

Las tres hipótesis planteadas fueron respaldadas dentro de este dataset.

**Limitación:** al tratarse de datos sintéticos, estos resultados corresponden al ejercicio académico y no deben utilizarse como estimaciones reales de Argentina.


## 11. Descarga de resultados

Estas celdas permiten descargar desde Colab el dataset limpio y, si se desea, todos los gráficos generados en un archivo ZIP.


In [ ]:
# Descargo el dataset limpio
from google.colab import files
files.download("df_clean.csv")


In [ ]:
# Opcional: comprimo y descargo todos los graficos
import shutil

shutil.make_archive("graficos_brecha_digital", "zip", "graficos")
files.download("graficos_brecha_digital.zip")
